# BarkSense — DS-CNN Training

Classifies 1-second audio windows into **bark / growl / grunt / ambient**.

Prerequisites:
```bash
pip install -r ../requirements.txt
python ../scripts/preprocess.py   # writes model/{train,val,test}.npz
```

In [ ]:
import itertools
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight

# ── Config ─────────────────────────────────────────────────────────────────────────
ALPHA   = 0.5    # DS-CNN width multiplier (mid-size for 120-dim log-mel features)
EPOCHS  = 50
BATCH   = 32
LR      = 1e-3
SEED    = 42
CLASSES = ['bark', 'growl', 'grunt', 'ambient']

ROOT      = Path('..')
MODEL_DIR = ROOT / 'model'
MODEL_DIR.mkdir(exist_ok=True)

tf.random.set_seed(SEED)
np.random.seed(SEED)
print(f'TensorFlow {tf.__version__}')

In [ ]:
# ── Load features ────────────────────────────────────────────────────────────────────
def load_split(name):
    d = np.load(str(MODEL_DIR / f'{name}.npz'))
    X = d['features'][..., np.newaxis].astype(np.float32)  # (N, n_mfcc, T, 1)
    y = d['labels'].astype(np.int32)
    return X, y

X_train, y_train = load_split('train')
X_val,   y_val   = load_split('val')
X_test,  y_test  = load_split('test')

# Z-score normalization — computed on training set only
mean = X_train.mean(axis=(0, 2), keepdims=True)
std  = X_train.std(axis=(0, 2),  keepdims=True) + 1e-8
X_train = (X_train - mean) / std
X_val   = (X_val   - mean) / std
X_test  = (X_test  - mean) / std
np.savez(str(MODEL_DIR / 'norm_stats.npz'), mean=mean, std=std)

INPUT_SHAPE = X_train.shape[1:]
N_CLASSES   = len(CLASSES)

print(f'Input shape : {INPUT_SHAPE}')
print(f'Train       : {X_train.shape}  labels: {np.bincount(y_train, minlength=N_CLASSES)}')
print(f'Val         : {X_val.shape}    labels: {np.bincount(y_val,   minlength=N_CLASSES)}')
print(f'Test        : {X_test.shape}   labels: {np.bincount(y_test,  minlength=N_CLASSES)}')

In [ ]:
# ── DS-CNN ───────────────────────────────────────────────────────────────────────────────────
def make_ds_cnn(input_shape, n_classes, alpha=1.0):
    """Depthwise-Separable CNN from the Hello Edge KWS paper."""
    def ch(n): return max(1, int(n * alpha))

    inp = tf.keras.Input(shape=input_shape, name='mfcc')
    x = tf.keras.layers.Conv2D(ch(64), (3, 3), padding='same', use_bias=False)(inp)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    for filters in [ch(64), ch(64), ch(128), ch(128)]:
        x = tf.keras.layers.DepthwiseConv2D((3, 3), padding='same', use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)
        x = tf.keras.layers.Conv2D(filters, (1, 1), use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)
        x = tf.keras.layers.Dropout(0.1)(x)

    x   = tf.keras.layers.GlobalAveragePooling2D()(x)
    out = tf.keras.layers.Dense(n_classes, activation='softmax')(x)
    return tf.keras.Model(inp, out, name=f'ds_cnn_a{alpha}')

model = make_ds_cnn(INPUT_SHAPE, N_CLASSES, alpha=ALPHA)
model.summary()

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# Class weights — growl/grunt likely under-represented relative to bark
present = np.unique(y_train)
weights = compute_class_weight('balanced', classes=present, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(present, weights)}
print(f'Class weights: {class_weight}')

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=12, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        str(MODEL_DIR / f'ds_cnn_a{ALPHA}_best.h5'),
        monitor='val_loss', save_best_only=True, verbose=0),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH,
    class_weight=class_weight,
    callbacks=callbacks,
)

In [ ]:
# ── Training curves ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'],         label='train')
axes[0].plot(history.history['val_loss'],     label='val')
axes[0].set_title('Loss');     axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history.history['accuracy'],     label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(MODEL_DIR / f'training_a{ALPHA}.png'), dpi=120)
plt.show()

In [ ]:
# ── Evaluate: confusion matrix + macro F1 + per-class recall ──────────────
y_pred   = np.argmax(model.predict(X_test, verbose=0), axis=1)
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f'Macro F1: {macro_f1:.4f}\n')
print(classification_report(y_test, y_pred, target_names=CLASSES, labels=list(range(N_CLASSES)), zero_division=0))

print('Per-class recall:')
for i, cls in enumerate(CLASSES):
    mask   = y_test == i
    recall = np.mean(y_pred[mask] == i) if mask.any() else float('nan')
    print(f'  {cls:10s}  {recall:.3f}  (n={mask.sum()})')

cm = confusion_matrix(y_test, y_pred, labels=list(range(N_CLASSES)))
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
ticks = range(N_CLASSES)
ax.set_xticks(ticks); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticks(ticks); ax.set_yticklabels(CLASSES)
thresh = cm.max() / 2 if cm.max() > 0 else 0.5
for i, j in itertools.product(ticks, ticks):
    ax.text(j, i, cm[i, j], ha='center', va='center',
            color='white' if cm[i, j] > thresh else 'black')
ax.set_ylabel('True'); ax.set_xlabel('Predicted')
ax.set_title(f'Confusion matrix — α={ALPHA}  macro-F1={macro_f1:.3f}')
plt.tight_layout()
plt.savefig(str(MODEL_DIR / f'confusion_a{ALPHA}.png'), dpi=120)
plt.show()

In [ ]:
# ── Save + baseline TFLite ────────────────────────────────────────────────────────────────
h5_path = MODEL_DIR / f'ds_cnn_a{ALPHA}_float32.h5'
model.save(str(h5_path))
print(f'Keras model → {h5_path}')

conv         = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = conv.convert()
tflite_path  = MODEL_DIR / f'ds_cnn_a{ALPHA}_float32.tflite'
tflite_path.write_bytes(tflite_bytes)
print(f'TFLite float32 → {tflite_path}  ({len(tflite_bytes)/1024:.1f} KB)')

interp = tf.lite.Interpreter(model_content=tflite_bytes)
interp.allocate_tensors()
inp_d, out_d = interp.get_input_details()[0], interp.get_output_details()[0]

tfl_preds = []
for x in X_test:
    interp.set_tensor(inp_d['index'], x[np.newaxis])
    interp.invoke()
    tfl_preds.append(int(np.argmax(interp.get_tensor(out_d['index']))))

tfl_acc = np.mean(np.array(tfl_preds) == y_test)
print(f'TFLite accuracy: {tfl_acc:.4f}  (Keras: {np.mean(y_pred == y_test):.4f})')
print('\nNext: python ../scripts/compress.py')